# FL Compression Study — Remaining Seeds

Runs only the configs that have **no complete seed yet**, using checkpoint resume where possible.

| Config | Checkpoint | Rounds needed | Time estimate |
|--------|-----------|---------------|---------------|
| sz_schedule_b s0 | round 20 ✓ | ~80 more rounds | ~6.7 hrs |
| sz_schedule_c s0 | **fresh start** (buggy ckpt discarded) | 100 rounds | ~8.3 hrs |
| sz_schedule_a s1 | round 21 ✓ | ~79 more rounds | ~6.6 hrs *(optional)* |

**SZ configs use `--local-epochs 1`** to cut ~10 min/round → ~5 min/round.

**Run order:** schedule_b → schedule_c → schedule_a s1 (optional)

**Output:** `results/resnet20_cifar10_remaining.csv` (appended — safe to re-run)

---
> The run auto-resumes from Drive checkpoints on Colab disconnect. Just re-run Cell 3.

## Cell 1 — Install dependencies
**Run once, then restart runtime.**

In [ ]:
!pip install -q 'flwr[simulation]==1.9.0' ray pandas
!pip install -q 'protobuf>=4.23' --force-reinstall
!pip install -q pysz

# flwr 1.9.0 uses np.float_ which was removed in numpy 2.0 — patch it in-place
!find /usr/local/lib /usr/lib -path "*/flwr/common/typing.py" 2>/dev/null \
    -exec sed -i 's/np\.float_/np.float64/g' {} \; -print

print('Done. Restart runtime now (Runtime → Restart runtime), then run Cell 2.')

## Cell 2 — Mount Drive and clone/pull repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys, shutil

REPO_URL   = 'https://github.com/ayananyways/fl-compression-study.git'
REPO_DIR   = '/content/fl-compression-study'
DRIVE_BASE = '/content/drive/MyDrive/fl_results'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'fl-flower'))

# Link results and checkpoints to Drive so they survive disconnects
for folder in ['results', 'checkpoints']:
    os.makedirs(f'{DRIVE_BASE}/{folder}', exist_ok=True)
    local = f'{REPO_DIR}/{folder}'
    drive_path = f'{DRIVE_BASE}/{folder}'
    if not os.path.islink(local):
        if os.path.exists(local):
            shutil.rmtree(local)
        os.symlink(drive_path, local)

print('Repo ready. Checkpoints and results saved to Drive.')
print('Working dir:', os.getcwd())

# ── Delete the buggy Schedule C checkpoint so it starts fresh ─────────────────
import glob
ckpt_dir = f'{REPO_DIR}/checkpoints/flower_cifar10_adaptive'
buggy = glob.glob(f'{ckpt_dir}/sz_schedule_c_s0_*.pkl')
if buggy:
    for f in buggy:
        os.remove(f)
    print(f'  Removed {len(buggy)} buggy Schedule C checkpoint(s) — will start fresh.')
else:
    print('  No Schedule C checkpoints found (clean slate).')

## Cell 3 — Run remaining configs

**Runs sequentially; each auto-resumes from checkpoint if Colab disconnects.**  
Re-run this cell after reconnecting — it picks up where it left off.

**Time budget per config (--local-epochs 1, ~5 min/round):**
- sz_schedule_b s0: ~80 rounds remaining → ~6.7 hrs
- sz_schedule_c s0 (fresh): 100 rounds → ~8.3 hrs
- sz_schedule_a s1 (optional): ~79 rounds remaining → ~6.6 hrs

In [ ]:
# ── DEBUG: run one config and print its full output ───────────────────────────
import subprocess, sys, os

REPO_DIR = '/content/fl-compression-study'
PYTHON   = sys.executable
RUN_PY   = f'{REPO_DIR}/fl-flower/run.py'
OUTPUT   = f'{REPO_DIR}/results/resnet20_cifar10_remaining.csv'

cmd = [
    PYTHON, RUN_PY,
    '--dataset',      'cifar10',
    '--rounds',       '3',          # just 3 rounds to expose the error fast
    '--num-clients',  '10',
    '--alpha',        '0.5',
    '--local-epochs', '1',
    '--compressor',   'sz',
    '--schedule',     'B',
    '--seed',         '0',
    '--output',       OUTPUT,
]

print('CMD:', ' '.join(cmd))
print()
result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
print('=== STDOUT ===')
print(result.stdout[-3000:] if result.stdout else '(empty)')
print('=== STDERR ===')
print(result.stderr[-3000:] if result.stderr else '(empty)')
print('=== EXIT CODE:', result.returncode, '===')

In [ ]:
import os, subprocess, sys, tempfile
from datetime import datetime

REPO_DIR = '/content/fl-compression-study'
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/fl-flower')

PYTHON = sys.executable
RUN_PY = f'{REPO_DIR}/fl-flower/run.py'
OUTPUT = f'{REPO_DIR}/results/resnet20_cifar10_remaining.csv'

# ── Pre-flight checks ─────────────────────────────────────────────────────────
print("Pre-flight checks...")
ok = True

try:
    import pysz
    print("  ✓ pysz")
except ImportError as e:
    print(f"  ✗ pysz missing: {e}")
    print("    → Run Cell 1 (install), restart runtime, then re-run Cell 2 and Cell 3.")
    ok = False

if not os.path.exists(RUN_PY):
    print(f"  ✗ run.py not found — run Cell 2 first")
    ok = False
else:
    print(f"  ✓ run.py found")

if not os.path.exists(f'{REPO_DIR}/fl-flower/adaptive_strategy.py'):
    print("  ✗ adaptive_strategy.py missing — Cell 2 git pull may have failed")
    ok = False
else:
    print("  ✓ adaptive_strategy.py found")

if not ok:
    raise SystemExit("Fix the errors above before running experiments.")

print("All checks passed.\n")

# ── Config list ───────────────────────────────────────────────────────────────
SZ_BASE = [
    '--dataset',      'cifar10',
    '--rounds',       '100',
    '--num-clients',  '10',
    '--alpha',        '0.5',
    '--local-epochs', '1',    # 1 epoch/round ≈ 5 min/round on Colab (vs 10 min at 2)
    '--output',       OUTPUT,
]

run_optional = True  # set False to skip sz_schedule_a s1

CONFIGS = [
    (
        'sz_schedule_b s0',
        ['--compressor', 'sz', '--schedule', 'B', '--seed', '0'],
        'CRITICAL — resumes from round 20 checkpoint (~80 rounds left, ~6.7 hrs)',
        False,
    ),
    (
        'sz_schedule_c s0 (FIXED)',
        ['--compressor', 'sz', '--schedule', 'C', '--seed', '0'],
        'CRITICAL — fresh start, fixed plateau trigger (100 rounds, ~8.3 hrs)',
        False,
    ),
    (
        'sz_schedule_a s1',
        ['--compressor', 'sz', '--schedule', 'A', '--seed', '1'],
        'Optional — second seed, resumes from round 21 (~79 rounds left, ~6.6 hrs)',
        True,
    ),
]

# ── Run ───────────────────────────────────────────────────────────────────────
total = sum(1 for *_, opt in CONFIGS if not opt or run_optional)
idx   = 0

for name, extra_flags, desc, optional in CONFIGS:
    if optional and not run_optional:
        print(f'  Skipping optional: {name}')
        continue
    idx += 1
    print(f'\n{"="*65}')
    print(f'[{idx}/{total}]  {name}')
    print(f'        {desc}')
    print(f'        Started: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    print(f'{"="*65}')

    cmd = [PYTHON, RUN_PY] + SZ_BASE + extra_flags

    # stderr goes to a temp file so we can show it on failure
    # stdout streams directly to the cell so progress is visible
    with tempfile.NamedTemporaryFile(mode='w', suffix='.err', delete=False) as ef:
        errpath = ef.name

    with open(errpath, 'w') as ef:
        result = subprocess.run(cmd, cwd=REPO_DIR, stderr=ef)

    if result.returncode != 0:
        print(f'\n  *** {name} FAILED (exit code {result.returncode})')
        with open(errpath) as ef:
            err = ef.read().strip()
        if err:
            print('  --- STDERR (last 2000 chars) ---')
            print(err[-2000:])
            print('  --------------------------------')
        print('  Re-run this cell to resume from the last checkpoint.')
    else:
        print(f'  ✓ {name} DONE at {datetime.now().strftime("%H:%M:%S")}')

    os.unlink(errpath)

print('\n' + '='*65)
print('All selected configs finished.')
print('Output:', OUTPUT)

## Cell 4 — Quick results summary

In [ ]:
import pandas as pd, numpy as np, os

OUTPUT = '/content/fl-compression-study/results/resnet20_cifar10_remaining.csv'

if not os.path.exists(OUTPUT):
    print('No output file yet — run Cell 3 first.')
else:
    # Handle mixed column counts (13-col old, 22-col new+adaptive)
    with open(OUTPUT) as f:
        lines = f.readlines()

    hdr13 = lines[0].strip().split(',')
    hdr22 = ('timestamp,round,compressor,seed,num_clients,alpha,'
             'val_accuracy,val_loss,bytes_sent,compression_ratio,'
             'compress_time_s,decompress_time_s,'
             'usnr_alpha,usnr_eb_mean,usnr_eb_min,usnr_eb_max,'
             'usnr_rms_mean,usnr_rms_min,usnr_rms_max,'
             'compressed_tensors_count,uncompressed_tensors_count,current_eb').split(',')

    rows = []
    for ln in lines[1:]:
        ln = ln.strip()
        if not ln: continue
        parts = ln.split(',')
        if len(parts) == len(hdr13):
            rows.append(dict(zip(hdr13, parts)))
        elif len(parts) == len(hdr22):
            rows.append(dict(zip(hdr22, parts)))

    df = pd.DataFrame(rows)
    for c in ['round', 'val_accuracy', 'bytes_sent', 'compression_ratio', 'compress_time_s']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    print(f'Total rows parsed: {len(df)}')
    print()

    rows_out = []
    for (comp, seed), grp in df.groupby(['compressor', 'seed']):
        max_r  = int(grp['round'].max())
        peak   = grp['val_accuracy'].max()
        last5  = grp.nlargest(5, 'round')['val_accuracy'].mean()
        ratio  = grp[grp['round'] > 0]['compression_ratio'].mean()
        mb     = grp[grp['round'] > 0]['bytes_sent'].mean() / 10 / 1e6
        rows_out.append({
            'config': comp, 'seed': seed,
            'max_round': max_r,
            'peak_acc': round(peak, 2),
            'last5_mean': round(last5, 2),
            'ratio': round(ratio, 2),
            'MB_per_round': round(mb, 3),
        })

    out = pd.DataFrame(rows_out).sort_values(['config', 'seed'])
    print('=== Remaining seeds summary ===')
    print(out.to_string(index=False))

    # Show schedule C plateau triggers if present
    if 'current_eb' in df.columns:
        sc = df[df['compressor'] == 'sz_schedule_c'].copy()
        if not sc.empty:
            sc['current_eb'] = pd.to_numeric(sc['current_eb'], errors='coerce')
            triggers = sc[sc['current_eb'].diff() > 0]
            if not triggers.empty:
                print()
                print('Schedule C plateau triggers:')
                for _, row in triggers.iterrows():
                    print(f"  round {int(row['round'])}: eb → {row['current_eb']}")
            else:
                print('\nSchedule C: no eb steps triggered yet.')

## Cell 5 — Convergence plot for remaining configs

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd, numpy as np, os

OUTPUT   = '/content/fl-compression-study/results/resnet20_cifar10_remaining.csv'
PLOT_OUT = '/content/fl-compression-study/results/remaining_convergence.png'

if not os.path.exists(OUTPUT):
    print('No output file yet.')
else:
    with open(OUTPUT) as f:
        lines = f.readlines()

    hdr13 = lines[0].strip().split(',')
    hdr22 = ('timestamp,round,compressor,seed,num_clients,alpha,'
             'val_accuracy,val_loss,bytes_sent,compression_ratio,'
             'compress_time_s,decompress_time_s,'
             'usnr_alpha,usnr_eb_mean,usnr_eb_min,usnr_eb_max,'
             'usnr_rms_mean,usnr_rms_min,usnr_rms_max,'
             'compressed_tensors_count,uncompressed_tensors_count,current_eb').split(',')

    rows = []
    for ln in lines[1:]:
        ln = ln.strip()
        if not ln: continue
        parts = ln.split(',')
        if len(parts) == len(hdr13):
            rows.append(dict(zip(hdr13, parts)))
        elif len(parts) == len(hdr22):
            rows.append(dict(zip(hdr22, parts)))

    df = pd.DataFrame(rows)
    df['round'] = pd.to_numeric(df['round'], errors='coerce')
    df['val_accuracy'] = pd.to_numeric(df['val_accuracy'], errors='coerce')

    PALETTE = {
        'sz_schedule_a': '#1a9641',
        'sz_schedule_b': '#66bd63',
        'sz_schedule_c': '#006837',
    }

    fig, ax = plt.subplots(figsize=(11, 5))
    for (comp, seed), grp in df.groupby(['compressor', 'seed']):
        grp = grp.sort_values('round')
        rm  = grp['val_accuracy'].rolling(5, min_periods=1).mean()
        color = PALETTE.get(comp, '#888')
        label = f'{comp} s{seed}'
        ax.plot(grp['round'], rm, color=color, lw=2, label=label)

    ax.set_xlabel('Communication round', fontsize=12)
    ax.set_ylabel('Validation accuracy (%)', fontsize=12)
    ax.set_title('Remaining seeds — SZ adaptive schedules (fixed Schedule C)', fontsize=11)
    ax.legend(fontsize=9, loc='lower right')
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(PLOT_OUT, dpi=150)
    plt.show()
    print('Saved:', PLOT_OUT)